In [0]:
df = spark.read.table("ecommerce_analytics.bronze.sales")
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col

product_schema = StructType([
    StructField("curr", StringType()),
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("price", StringType()),
    StructField("qty", StringType()),
    StructField("unit", StringType())
    ])


df_parsed_product = df.withColumn("product", from_json(col("product"), product_schema))

sales_df = df_parsed_product.select(
    "customer_id",
    "customer_name",
    "order_date",
    "product_name",
    "product_category",
    col("product.id").alias("product_id"),
    col("product.name").alias("product_description"),
    col("product.price").alias("price"),
    col("product.qty").alias("qty"),
    col("product.unit").alias("unit"),
    "total_price",
    col("product.curr").alias("curr")
)


sales_df.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.sales")





########################################
########--SCD-1 IMPLEMENTATION----##########
########################################

def scd_merge_table(spark, source_table, target_table, business_key): 
     # Take new data (source) and merge into existing table (target 

    if not spark.catalog.tableExists(target_table):                                    #check if table exists
        print("First Load: Creating Silver Table", target_table)
        source_table.write.format("delta").mode("overwrite").saveAsTable(target_table)
        #creates table (Initial load , {full load})
        print("Table Created")

    else:
        print("Incremental Load: Performing SCD Type 1 Merge")

        delta_table = DeltaTable.forName(spark, target_table)   #Load existing table as  delta table (need for merger) 

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]     #for us  (target.customer_id = source.customer_id)
        )                                                                #this define show much record matches


        delta_table.alias("target").merge(source_table.alias("source"),    #join source + target
                                        merge_condition
                                        ).whenMatchedUpdateAll()\          # if record exists update all
                                            .whenNotMatchedInsertAll()\    # if new INSERT
                                            .execute()                     # runs the merge
        print("Merge Successfully Completed")


In [0]:
{"curr":"USD","id":"AVpiE9hhilAPnD_xAfSU","name":"Cyber-shot DSC-RX100 V Digital Camera","price":2798,"qty":4,"unit":"pcs"}

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col

product_schema = StructType([
    StructField("curr", StringType()),
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("price", StringType()),
    StructField("qty", StringType()),
    StructField("unit", StringType())
    ])


df_parsed_product = df.withColumn("product", from_json(col("product"), product_schema))
df_parsed_product.display()


In [0]:
df_explode_product.printSchema()

In [0]:
sales_df = df_parsed_product.select(
    "customer_id",
    "customer_name",
    "order_date",
    "product_name",
    "product_category",
    col("product.id").alias("product_id"),
    col("product.name").alias("product_description"),
    col("product.price").alias("price"),
    col("product.qty").alias("qty"),
    col("product.unit").alias("unit"),
    "total_price",
    col("product.curr").alias("curr")
)
sales_df.display()

sales_df.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.sales")
